# 05 - Temporal Baseline

## Objetivo

Avaliar baselines de ranking usando o dataset temporal produzido no notebook `04-feature-engineering.ipynb`.

## Inputs

- `data/features/temporal_modeling_dataset_v1/`
- `data/processed/orders_product_unified.parquet`

## Regras de avaliacao

- Nao ha split aleatorio por linha.
- Os splits `train`, `validation` e `test` vem do dataset temporal.
- O ranking e feito dentro de cada `user_window_id`.
- `target` e usado apenas para avaliacao.
- As metricas sao calculadas em K = 5, 10, 20.
- `recall_local@K` mede a qualidade do ranking dentro dos produtos que chegaram ao dataset de candidatos.
- `recall_global@K` mede a qualidade do sistema completo `candidate generation + ranking`.
- Neste notebook, `recall_global@K` e mantido porque os baselines estao fortemente ligados a ordem e as fontes da geracao de candidatos.
- `recall_global@K` nao sera usado como metrica principal para escolher o melhor ranker; ele serve para quantificar o impacto pratico da cobertura dos candidatos no carrinho real completo.

## Recall local e recall global

Este notebook separa duas leituras de Recall@K:

- `recall_local`: avalia a qualidade do modelo de ranking. O denominador e apenas a quantidade de produtos positivos que entraram no conjunto de candidatos da janela.
- `recall_global`: avalia a qualidade do sistema completo. O denominador e o tamanho real do carrinho alvo completo, mesmo quando alguns produtos nao foram gerados como candidatos.

Na pratica:

```text
recall_local_at_k = hits_at_k / positivos_cobertos_pelos_candidatos
recall_global_at_k = hits_at_k / produtos_do_carrinho_real_completo
```

O `recall_global` e limitado pelo recall ceiling da geracao de candidatos medido no notebook 03. O `recall_local` isola melhor a qualidade do ranker quando o universo de candidatos e fixo.

## Metrica norteadora

A metrica principal deste notebook e `ndcg@10`, seguida por `hit_rate@10`, `recall_local@10`, `recall_global@10` e `precision@10`.

A escolha de `ndcg` como metrica norteadora e intencional: recomendacao e um problema de ranking. Nao basta acertar produtos relevantes; e importante coloca-los nas primeiras posicoes do Top K. `hit_rate` mede se pelo menos um produto util apareceu, `recall_local` mede a recuperacao do ranker dentro dos candidatos, `recall_global` mede o sistema ponta a ponta, e `precision` fica como leitura complementar de densidade de acertos no Top K.


---

## 1. Setup inicial


In [1]:
import os
import subprocess
import tempfile
import time
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from dotenv import load_dotenv

pd.set_option("display.max_columns", None)

In [2]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"

UNIFIED_DATASET_PATH = PROCESSED_DIR / "orders_product_unified.parquet"
TEMPORAL_MODELING_DATASET_DIR = FEATURES_DIR / "temporal_modeling_dataset_v1"
TEMPORAL_MODELING_DATASET_DVC_PATH = FEATURES_DIR / "temporal_modeling_dataset_v1.dvc"

K_VALUES = [5, 10, 20]
EVALUATION_SPLITS = ["train", "validation", "test"]
PROGRESS_EVERY_PARTITIONS = 5
PRIMARY_K = 10
PRIMARY_METRIC = "ndcg"
RUN_MLFLOW = False
EXPERIMENT_NAME = "mlp-temporal-baseline-helio-v1"

for input_path in [
    UNIFIED_DATASET_PATH,
    TEMPORAL_MODELING_DATASET_DIR,
    TEMPORAL_MODELING_DATASET_DVC_PATH,
]:
    assert input_path.exists(), f"Entrada nao encontrada: {input_path}"

part_paths = sorted(TEMPORAL_MODELING_DATASET_DIR.glob("*.parquet"))
assert part_paths, (
    f"Nenhuma particao Parquet encontrada em: {TEMPORAL_MODELING_DATASET_DIR}"
)

print(f"Dataset temporal encontrado: {TEMPORAL_MODELING_DATASET_DIR}")
print(f"Ponteiro DVC encontrado: {TEMPORAL_MODELING_DATASET_DVC_PATH}")
print(f"Dataset unificado encontrado: {UNIFIED_DATASET_PATH}")
print(f"Particoes: {len(part_paths):,}")

Dataset temporal encontrado: C:\Users\erick\projetos\mlp-market-recommender-system\data\features\temporal_modeling_dataset_v1
Ponteiro DVC encontrado: C:\Users\erick\projetos\mlp-market-recommender-system\data\features\temporal_modeling_dataset_v1.dvc
Dataset unificado encontrado: C:\Users\erick\projetos\mlp-market-recommender-system\data\processed\orders_product_unified.parquet
Particoes: 47


### 1.1 Schema sem carregar o dataset completo


In [3]:
modeling_dataset = ds.dataset(TEMPORAL_MODELING_DATASET_DIR, format="parquet")
dataset_columns = modeling_dataset.schema.names

required_columns = {
    "split",
    "user_id",
    "product_id",
    "user_window_id",
    "target_order_id",
    "target",
    "candidate_source",
    "candidate_rank",
    "history_group",
    "history_order_count",
    "history_unique_products",
    "user_product_was_bought_before",
    "user_product_purchase_count",
    "user_product_reorder_count",
    "user_product_orders_since_last_purchase",
    "user_product_days_since_last_purchase",
    "user_product_purchase_share",
    "user_aisle_purchase_count",
    "user_department_purchase_count",
}

missing_columns = sorted(required_columns - set(dataset_columns))
assert not missing_columns, f"Colunas obrigatorias ausentes: {missing_columns}"

print(f"Colunas no dataset temporal: {len(dataset_columns):,}")
print(dataset_columns)

Colunas no dataset temporal: 32
['user_id', 'product_id', 'split', 'window_number', 'user_window_id', 'target_order_id', 'target_order_number', 'history_start_order_number', 'history_end_order_number', 'history_order_count', 'history_unique_products', 'history_group', 'candidate_source', 'candidate_rank', 'target', 'aisle_id', 'department_id', 'user_prior_order_count', 'user_avg_basket_size', 'user_avg_days_between_orders', 'user_reorder_rate', 'user_total_items', 'user_has_single_prior_order', 'user_product_purchase_count', 'user_product_reorder_count', 'user_product_avg_add_to_cart_order', 'user_product_orders_since_last_purchase', 'user_product_days_since_last_purchase', 'user_product_was_bought_before', 'user_product_purchase_share', 'user_aisle_purchase_count', 'user_department_purchase_count']


In [4]:
sample_columns = [
    "split",
    "user_id",
    "user_window_id",
    "target_order_id",
    "product_id",
    "target",
    "candidate_source",
    "candidate_rank",
    "user_product_was_bought_before",
    "user_product_purchase_count",
]

sample_df = pd.read_parquet(part_paths[0], columns=sample_columns).head(10)

sample_df

,split,user_id,user_window_id,target_order_id,product_id,target,candidate_source,candidate_rank,user_product_was_bought_before,user_product_purchase_count
0,train,1,1_train_3108588,3108588,196,1,recompra,1,1,7
1,train,1,1_train_3108588,3108588,12427,1,recompra,2,1,7
2,train,1,1_train_3108588,3108588,10258,1,recompra,3,1,6
3,train,1,1_train_3108588,3108588,25133,1,recompra,4,1,5
4,train,1,1_train_3108588,3108588,13032,0,recompra,5,1,2
5,train,1,1_train_3108588,3108588,13176,0,recompra,6,1,2
6,train,1,1_train_3108588,3108588,26405,0,recompra,7,1,2
7,train,1,1_train_3108588,3108588,26088,0,recompra,8,1,2
8,train,1,1_train_3108588,3108588,10326,0,recompra,9,1,1
9,train,1,1_train_3108588,3108588,17122,0,recompra,10,1,1


### 1.2 Contagem incremental por split


In [5]:
split_stats = []

for part_path in part_paths:
    part_df = pd.read_parquet(
        part_path,
        columns=["split", "user_window_id", "user_id", "target"],
    )

    stats = part_df.groupby("split", as_index=False).agg(
        rows=("target", "size"),
        windows=("user_window_id", "nunique"),
        users=("user_id", "nunique"),
        positives=("target", "sum"),
    )
    split_stats.append(stats)

    del part_df

split_stats_df = (
    pd.concat(split_stats, ignore_index=True).groupby("split", as_index=False).sum()
)
split_stats_df["positive_rate"] = split_stats_df["positives"] / split_stats_df["rows"]

split_stats_df

,split,rows,windows,users,positives,positive_rate
0,test,23181800,115909,115909,866485,0.037378
1,train,46363600,231818,115909,1740928,0.037549
2,validation,23181800,115909,115909,847423,0.036556


---

## 2. Metricas de ranking

A avaliacao separa duas perguntas diferentes.

A primeira pergunta e sobre o ranker: dado o conjunto de candidatos disponivel para uma janela, a ordenacao coloca os produtos relevantes no topo? Para isso, as metricas principais sao `ndcg@K`, `hit_rate@K`, `recall_local@K` e `precision@K`.

A segunda pergunta e sobre o sistema completo: considerando o carrinho real inteiro do `target_order_id`, quanto o pipeline `candidate generation + ranking` consegue recuperar no top K? Para isso, este notebook tambem calcula `recall_global@K`.

`recall_global@K` depende diretamente da cobertura da etapa de candidatos. Portanto, ele nao deve ser interpretado como uma metrica pura de ranking. Ele e mantido aqui porque o notebook de baseline tambem documenta a qualidade operacional da estrategia de candidatos antes do MLP.

`ndcg@K` e calculado somente nas janelas com pelo menos um positivo coberto pelos candidatos. Isso evita punir o ranker por produtos que nunca chegaram ate ele.


In [6]:
orders_dataset = ds.dataset(UNIFIED_DATASET_PATH, format="parquet")


def load_target_product_counts(target_order_ids):
    target_order_ids = [int(order_id) for order_id in target_order_ids]

    table = orders_dataset.to_table(
        columns=["order_id", "product_id"],
        filter=ds.field("order_id").isin(target_order_ids),
    )
    target_products_df = table.to_pandas()

    assert not target_products_df.empty, "Nenhum produto de target encontrado."

    target_counts = target_products_df.groupby("order_id", as_index=False).agg(
        global_target_count=("product_id", "nunique")
    )

    return target_counts


def dcg_at_k(targets):
    targets = np.asarray(targets, dtype=float)
    if len(targets) == 0:
        return 0.0

    discounts = 1.0 / np.log2(np.arange(2, len(targets) + 2))

    return float(np.sum(targets * discounts))


def evaluate_ranked_frame(scored_df, score_column, k_values, target_count_by_window):
    ranked_df = scored_df.sort_values(
        ["user_window_id", score_column, "product_id"],
        ascending=[True, False, True],
        kind="mergesort",
    ).copy()

    ranked_df["rank"] = ranked_df.groupby("user_window_id").cumcount().add(1)

    local_target_counts = ranked_df.groupby("user_window_id")["target"].sum()
    global_target_counts = target_count_by_window.reindex(local_target_counts.index)

    assert global_target_counts.notna().all(), (
        "Existem janelas sem denominador global de target."
    )

    global_windows = global_target_counts[global_target_counts > 0].index
    local_windows = local_target_counts[local_target_counts > 0].index

    ranked_df = ranked_df[ranked_df["user_window_id"].isin(global_windows)]

    rows = []

    for k in k_values:
        top_k = ranked_df[ranked_df["rank"] <= k]
        per_window_hits = top_k.groupby("user_window_id")["target"].sum()
        per_window_hits = per_window_hits.reindex(global_windows, fill_value=0)

        global_denominator = global_target_counts.loc[global_windows]
        local_denominator = local_target_counts.loc[local_windows]
        local_hits = per_window_hits.reindex(local_windows, fill_value=0)

        ndcg_values = []
        local_ranked_df = ranked_df[ranked_df["user_window_id"].isin(local_windows)]
        for window_id, window_targets in local_ranked_df.groupby("user_window_id")[
            "target"
        ]:
            observed = window_targets.head(k).to_numpy()
            ideal_hits = int(min(local_target_counts.loc[window_id], k))
            ideal = np.ones(ideal_hits, dtype=float)
            ideal_dcg = dcg_at_k(ideal)
            ndcg_values.append(dcg_at_k(observed) / ideal_dcg if ideal_dcg > 0 else 0.0)

        rows.append(
            {
                "k": k,
                "evaluated_windows": len(global_windows),
                "local_evaluated_windows": len(local_windows),
                "precision": float((per_window_hits / k).mean()),
                "hit_rate": float((per_window_hits > 0).mean()),
                "recall_local": float((local_hits / local_denominator).mean()),
                "recall_global": float((per_window_hits / global_denominator).mean()),
                "ndcg": float(np.mean(ndcg_values)) if ndcg_values else 0.0,
            }
        )

    return pd.DataFrame(rows)

---

## 3. Scores dos baselines

Os baselines deste notebook sao heuristicas deterministicas de ranking. Eles nao treinam um classificador e nao usam `target` como feature.

A escolha foi proposital: o objetivo desta etapa e criar referencias interpretaveis e baratas para comparar futuros modelos treinaveis. Um `DummyClassifier` seria util para classificacao tabular, mas aqui o problema e ranking dentro de cada `user_window_id`; uma probabilidade constante ou estratificada nao produz uma ordenacao competitiva nem incorpora sinais de recomendacao.

Estrategias avaliadas:

- `ordem_gerador_candidatos`: respeita a ordem em que a estrategia de candidatos selecionou os produtos. Serve como baseline direto do candidate generation.
- `recompra_usuario`: prioriza produtos ja comprados pelo usuario, frequencia historica, recompra e recencia. Serve como baseline forte para grocery, onde recompra costuma dominar.
- `afinidade_categoria`: prioriza produtos de aisles/departments frequentes no historico da janela. Mede o valor de preferencia por categoria, especialmente para produtos novos.
- `heuristico_temporal`: combina recompra, recompra recorrente, afinidade de aisle, rank original do candidato e fonte do candidato. Serve como baseline manual mais forte antes do MLP.


In [7]:
BASELINE_COLUMNS = [
    "split",
    "user_window_id",
    "target_order_id",
    "user_id",
    "product_id",
    "target",
    "candidate_source",
    "candidate_rank",
    "history_group",
    "history_unique_products",
    "user_product_was_bought_before",
    "user_product_purchase_count",
    "user_product_reorder_count",
    "user_product_orders_since_last_purchase",
    "user_product_days_since_last_purchase",
    "user_product_purchase_share",
    "user_aisle_purchase_count",
    "user_department_purchase_count",
]

SOURCE_PRIORITY = {
    "recompra": 4,
    "similarity": 3,
    "category": 2,
    "global": 1,
}

BASELINE_SCORE_COLUMNS = {
    "ordem_gerador_candidatos": "score_candidate_rank",
    "recompra_usuario": "score_repurchase",
    "afinidade_categoria": "score_category_affinity",
    "heuristico_temporal": "score_temporal_heuristic",
}

In [8]:
def add_user_window_normalized_column(df, source_column, output_column):
    grouped = df.groupby("user_window_id")[source_column]
    min_values = grouped.transform("min")
    max_values = grouped.transform("max")
    denominator = (max_values - min_values).replace(0, 1)
    df[output_column] = (df[source_column] - min_values) / denominator

    return df


def add_baseline_scores(df):
    scored_df = df.copy()

    scored_df["candidate_source_priority"] = (
        scored_df["candidate_source"].map(SOURCE_PRIORITY).fillna(0)
    )
    scored_df["candidate_rank_inverse"] = -scored_df["candidate_rank"].astype(float)

    scored_df = add_user_window_normalized_column(
        scored_df,
        "user_product_purchase_count",
        "norm_user_product_purchase_count",
    )
    scored_df = add_user_window_normalized_column(
        scored_df,
        "user_product_reorder_count",
        "norm_user_product_reorder_count",
    )
    scored_df = add_user_window_normalized_column(
        scored_df,
        "user_aisle_purchase_count",
        "norm_user_aisle_purchase_count",
    )
    scored_df = add_user_window_normalized_column(
        scored_df,
        "candidate_rank_inverse",
        "norm_candidate_rank_inverse",
    )

    scored_df["score_candidate_rank"] = scored_df["candidate_rank_inverse"]
    scored_df["score_repurchase"] = (
        scored_df["user_product_was_bought_before"] * 10_000
        + scored_df["user_product_purchase_count"] * 100
        + scored_df["user_product_reorder_count"] * 10
        - scored_df["user_product_orders_since_last_purchase"]
    )
    scored_df["score_category_affinity"] = (
        scored_df["user_aisle_purchase_count"] * 100
        + scored_df["user_department_purchase_count"]
        + scored_df["candidate_source_priority"]
    )
    scored_df["score_temporal_heuristic"] = (
        0.45 * scored_df["norm_user_product_purchase_count"]
        + 0.25 * scored_df["norm_user_product_reorder_count"]
        + 0.15 * scored_df["norm_user_aisle_purchase_count"]
        + 0.10 * scored_df["norm_candidate_rank_inverse"]
        + 0.05 * (scored_df["candidate_source_priority"] / 4)
    )

    assert "target" not in [
        "score_candidate_rank",
        "score_repurchase",
        "score_category_affinity",
        "score_temporal_heuristic",
    ]

    return scored_df

---

## 4. Avaliacao incremental

A avaliacao e feita por particao para evitar carregar o dataset inteiro em memoria.

Cada particao e lida uma unica vez. Dentro dela, o notebook separa `train`, `validation` e `test`, calcula os scores uma vez e avalia todos os baselines para cada split disponivel. Isso evita reler a mesma particao por split ou por baseline.

A celula imprime progresso na primeira particao e depois a cada 5 particoes, incluindo linhas, janelas e tempo de cada split dentro da particao.

Para uma execucao exploratoria mais rapida, `EVALUATION_SPLITS` pode ser temporariamente ajustado para `["validation"]`. Para resultado final do notebook, mantenha `["train", "validation", "test"]`.


In [9]:
def build_target_count_by_window(part_df):
    window_targets = part_df[["user_window_id", "target_order_id"]].drop_duplicates()

    target_counts = load_target_product_counts(
        window_targets["target_order_id"].unique()
    )

    window_targets = window_targets.merge(
        target_counts,
        left_on="target_order_id",
        right_on="order_id",
        how="left",
    )

    assert window_targets["global_target_count"].notna().all(), (
        "Alguma janela ficou sem tamanho do carrinho alvo."
    )

    return window_targets.set_index("user_window_id")["global_target_count"]


def weighted_metric_average(group_df, metric_column):
    weight_column = (
        "local_evaluated_windows"
        if metric_column in ["ndcg", "recall_local"]
        else "evaluated_windows"
    )
    weights = group_df[weight_column]

    if weights.sum() == 0:
        return 0.0

    return float(np.average(group_df[metric_column], weights=weights))


def aggregate_partition_metrics(partition_metrics_df):
    metric_columns = ["precision", "hit_rate", "recall_local", "recall_global", "ndcg"]

    weighted_rows = []
    for keys, group_df in partition_metrics_df.groupby(["baseline", "split", "k"]):
        row = {
            "baseline": keys[0],
            "split": keys[1],
            "k": int(keys[2]),
            "evaluated_windows": int(group_df["evaluated_windows"].sum()),
            "local_evaluated_windows": int(group_df["local_evaluated_windows"].sum()),
        }
        for metric_column in metric_columns:
            row[metric_column] = weighted_metric_average(group_df, metric_column)
        weighted_rows.append(row)

    return pd.DataFrame(weighted_rows)


def evaluate_all_partitions():
    evaluation_start = time.perf_counter()
    metric_parts = []
    evaluation_splits = set(EVALUATION_SPLITS)

    for part_idx, part_path in enumerate(part_paths):
        partition_start = time.perf_counter()

        part_df = pd.read_parquet(part_path, columns=BASELINE_COLUMNS)
        part_df = part_df[part_df["split"].isin(evaluation_splits)].copy()

        if part_df.empty:
            continue

        scored_df = add_baseline_scores(part_df)
        partition_split_labels = []

        for split_name, split_df in scored_df.groupby("split", sort=False):
            split_start = time.perf_counter()
            target_count_by_window = build_target_count_by_window(split_df)

            for baseline_name, score_column in BASELINE_SCORE_COLUMNS.items():
                metrics_df = evaluate_ranked_frame(
                    scored_df=split_df,
                    score_column=score_column,
                    k_values=K_VALUES,
                    target_count_by_window=target_count_by_window,
                )
                metrics_df["baseline"] = baseline_name
                metrics_df["split"] = split_name
                metrics_df["partition"] = part_path.name
                metric_parts.append(metrics_df)

            partition_split_labels.append(
                f"{split_name}: {len(split_df):,} linhas, "
                f"{split_df['user_window_id'].nunique():,} janelas, "
                f"{time.perf_counter() - split_start:.1f}s"
            )

            del target_count_by_window

        elapsed = time.perf_counter() - partition_start
        if part_idx == 0 or (part_idx + 1) % PROGRESS_EVERY_PARTITIONS == 0:
            print(
                f"particao {part_idx + 1}/{len(part_paths)} | "
                f"{'; '.join(partition_split_labels)} | total={elapsed:.1f}s"
            )

        del part_df, scored_df

    if not metric_parts:
        return pd.DataFrame()

    partition_metrics_df = pd.concat(metric_parts, ignore_index=True)
    results_df = aggregate_partition_metrics(partition_metrics_df)

    print(f"Avaliacao finalizada em {time.perf_counter() - evaluation_start:.1f}s")

    return results_df

In [10]:
baseline_results_df = evaluate_all_partitions()

baseline_results_df.sort_values(["split", "k", "ndcg"], ascending=[True, True, False])

particao 1/47 | train: 986,800 linhas, 4,934 janelas, 5.0s; validation: 493,400 linhas, 2,467 janelas, 2.6s; test: 493,400 linhas, 2,467 janelas, 2.6s | total=11.3s


particao 5/47 | train: 986,800 linhas, 4,934 janelas, 5.1s; validation: 493,400 linhas, 2,467 janelas, 2.6s; test: 493,400 linhas, 2,467 janelas, 2.5s | total=11.3s


particao 10/47 | train: 986,400 linhas, 4,932 janelas, 5.3s; validation: 493,200 linhas, 2,466 janelas, 2.6s; test: 493,200 linhas, 2,466 janelas, 2.6s | total=11.5s


particao 15/47 | train: 986,400 linhas, 4,932 janelas, 5.3s; validation: 493,200 linhas, 2,466 janelas, 2.7s; test: 493,200 linhas, 2,466 janelas, 2.6s | total=11.9s


particao 20/47 | train: 986,400 linhas, 4,932 janelas, 5.0s; validation: 493,200 linhas, 2,466 janelas, 2.7s; test: 493,200 linhas, 2,466 janelas, 2.5s | total=11.4s


...

particao 35/47 | train: 986,400 linhas, 4,932 janelas, 5.0s; validation: 493,200 linhas, 2,466 janelas, 2.7s; test: 493,200 linhas, 2,466 janelas, 2.6s | total=11.4s


particao 40/47 | train: 986,400 linhas, 4,932 janelas, 5.4s; validation: 493,200 linhas, 2,466 janelas, 2.7s; test: 493,200 linhas, 2,466 janelas, 2.6s | total=11.8s


particao 45/47 | train: 986,400 linhas, 4,932 janelas, 5.3s; validation: 493,200 linhas, 2,466 janelas, 2.7s; test: 493,200 linhas, 2,466 janelas, 2.6s | total=11.8s


Avaliacao finalizada em 545.3s


,baseline,split,k,evaluated_windows,local_evaluated_windows,precision,hit_rate,recall_local,recall_global,ndcg
27,recompra_usuario,test,5,115909,112398,0.361097,0.791345,0.309870,0.232619,0.456151
18,ordem_gerador_candidatos,test,5,115909,112398,0.361094,0.791362,0.309870,0.232613,0.456146
9,heuristico_temporal,test,5,115909,112398,0.352746,0.781639,0.300242,0.225726,0.445405
0,afinidade_categoria,test,5,115909,112398,0.163899,0.532340,0.146751,0.111141,0.207952
19,ordem_gerador_candidatos,test,10,115909,112398,0.281881,0.863609,0.447413,0.333968,0.462666
28,recompra_usuario,test,10,115909,112398,0.281838,0.863540,0.447292,0.333867,0.462616
10,heuristico_temporal,test,10,115909,112398,0.274042,0.855059,0.430999,0.322121,0.449551
1,afinidade_categoria,test,10,115909,112398,0.125297,0.643781,0.204217,0.153435,0.210333
20,ordem_gerador_candidatos,test,20,115909,112398,0.204665,0.911154,0.604763,0.447633,0.508897
29,recompra_usuario,test,20,115909,112398,0.204249,0.910102,0.602373,0.445818,0.508047


In [11]:
# Persiste os resultados para consumo downstream (notebook 06 compara o MLP
# com o melhor baseline lendo este arquivo, em vez de valores hardcoded).
BASELINE_RESULTS_PATH = PROJECT_ROOT / "data" / "processed" / "baseline_results.parquet"
BASELINE_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
baseline_results_df.to_parquet(BASELINE_RESULTS_PATH, index=False)
print(f"Resultados dos baselines salvos em: {BASELINE_RESULTS_PATH}")

Resultados dos baselines salvos em: C:\Users\erick\projetos\mlp-market-recommender-system\data\processed\baseline_results.parquet


---

## 5. Comparacao no split de validacao

A escolha do baseline de referencia usa `ndcg@10` no split de validacao.

O split `test` permanece como holdout final offline. Ele deve ser usado para leitura final do sistema depois de escolher a estrategia/modelo com base em validacao.


In [12]:
validation_comparison = baseline_results_df[
    baseline_results_df["split"] == "validation"
].pivot_table(
    index="baseline",
    columns="k",
    values=["ndcg", "hit_rate", "recall_local", "recall_global", "precision"],
)

validation_comparison

hit_rate                          ndcg            \
k                               5         10        20        5         10   
baseline                                                                     
afinidade_categoria       0.541028  0.648138  0.734973  0.215757  0.217371   
heuristico_temporal       0.782485  0.856301  0.905745  0.449879  0.455739   
ordem_gerador_candidatos  0.791845  0.864618  0.912578  0.459596  0.467773   
recompra_usuario          0.791871  0.864489  0.911042  0.459598  0.467662   

                                   precision                      \
k                               20        5         10        20   
baseline                                                           
afinidade_categoria       0.237606  0.167851  0.126445  0.088176   
heuristico_temporal       0.499332  0.353046  0.273606  0.196397   
ordem_gerador_candidatos  0.514524  0.360833  0.280981  0.203020   
recompra_usuario          0.513254  0.360826  0.280888  0.202370   

                         recall_global                     recall_local  \
k                                   5         10        20           5    
baseline                                                                  
afinidade_categoria           0.115941  0.158337  0.211143     0.154336   
heuristico_temporal           0.230543  0.327875  0.434567     0.308718   
ordem_gerador_candidatos      0.237048  0.338734  0.451199     0.317947   
recompra_usuario              0.237049  0.338575  0.448722     0.317957   

                                              
k                               10        20  
baseline                                      
afinidade_categoria       0.212002  0.285998  
heuristico_temporal       0.441327  0.590231  
ordem_gerador_candidatos  0.456651  0.612685  
recompra_usuario          0.456416  0.609275

In [13]:
best_validation_baseline = (
    baseline_results_df[
        (baseline_results_df["split"] == "validation")
        & (baseline_results_df["k"] == PRIMARY_K)
    ]
    .sort_values(PRIMARY_METRIC, ascending=False)
    .iloc[0]
)

best_validation_baseline.to_frame().T

,baseline,split,k,evaluated_windows,local_evaluated_windows,precision,hit_rate,recall_local,recall_global,ndcg
25,ordem_gerador_candidatos,validation,10,115909,112240,0.280981,0.864618,0.456651,0.338734,0.467773


### 5.1 Leitura dos resultados atuais

No split de validacao, a melhor referencia por `ndcg@10` foi `ordem_gerador_candidatos`.

Resultado principal em validacao:

- `ordem_gerador_candidatos`: `ndcg@10 = 0.467768`, `hit_rate@10 = 0.864618`, `recall_local@10 = 0.456645`, `recall_global@10 = 0.338734`, `precision@10 = 0.280981`.
- `recompra_usuario`: desempenho praticamente empatado em `ndcg@10 = 0.467657`, indicando que a ordem de candidatos e o sinal de recompra carregam quase a mesma forca no baseline atual.
- `heuristico_temporal`: ficou um pouco abaixo (`ndcg@10 = 0.455734`), sugerindo que a combinacao manual adicionou sinais uteis, mas nao superou a ordem original/recompra.
- `afinidade_categoria`: ficou bem abaixo (`ndcg@10 = 0.217366`), funcionando mais como fallback de descoberta do que como baseline principal.

**Conclusao dos resultados de validacao**:

O melhor baseline por `ndcg@10` foi `ordem_gerador_candidatos`, mas a diferenca para `recompra_usuario` e marginal. Isso sugere que a ordem original dos candidatos esta fortemente alinhada ao historico de recompra do usuario.

O `hit_rate@10` alto mostra que, na maioria das janelas, pelo menos um produto relevante aparece no top 10. Porem, o `recall_local@10` em torno de 0.46 indica que menos da metade dos positivos cobertos pelos candidatos chega ao top 10. Isso deixa espaco para um modelo treinavel melhorar a ordenacao dentro de cada `user_window_id`.

O `recall_global@10` em torno de 0.34 deve ser lido como metrica do sistema completo `candidate generation + ranking`. Ele reflete tanto a cobertura limitada da geracao de candidatos quanto a ordenacao do baseline. Como o recall ceiling dos candidatos ja foi analisado no notebook 03, esta metrica aqui serve principalmente para documentar o impacto pratico da estrategia de candidatos no carrinho real completo.

Para a proxima etapa, o MLP deve ser comparado principalmente por `ndcg@10`, `hit_rate@10`, `recall_local@10` e `precision@10`. O `recall_global@10` pode ficar fora da rotina principal do notebook 06, porque mistura cobertura dos candidatos com qualidade do ranker.


---

## 6. Analise por segmento temporal


In [14]:
def evaluate_by_segment(segment_column, split_name, score_column):
    segment_parts = []

    for part_path in part_paths:
        part_df = pd.read_parquet(part_path, columns=BASELINE_COLUMNS)
        part_df = part_df[part_df["split"] == split_name].copy()

        if part_df.empty:
            continue

        target_count_by_window = build_target_count_by_window(part_df)
        scored_df = add_baseline_scores(part_df)

        for segment_value, segment_df in scored_df.groupby(segment_column):
            segment_target_count_by_window = target_count_by_window.reindex(
                segment_df["user_window_id"].unique()
            )
            metrics_df = evaluate_ranked_frame(
                scored_df=segment_df,
                score_column=score_column,
                k_values=K_VALUES,
                target_count_by_window=segment_target_count_by_window,
            )
            metrics_df["segment_column"] = segment_column
            metrics_df["segment_value"] = segment_value
            metrics_df["partition"] = part_path.name
            segment_parts.append(metrics_df)

        del part_df, scored_df, target_count_by_window

    segment_metrics_df = pd.concat(segment_parts, ignore_index=True)
    metric_columns = ["precision", "hit_rate", "recall_local", "recall_global", "ndcg"]

    rows = []
    for keys, group_df in segment_metrics_df.groupby(
        ["segment_column", "segment_value", "k"]
    ):
        row = {
            "segment_column": keys[0],
            "segment_value": keys[1],
            "k": int(keys[2]),
            "evaluated_windows": int(group_df["evaluated_windows"].sum()),
            "local_evaluated_windows": int(group_df["local_evaluated_windows"].sum()),
        }
        for metric_column in metric_columns:
            row[metric_column] = weighted_metric_average(group_df, metric_column)
        rows.append(row)

    return pd.DataFrame(rows)

In [15]:
validation_segment_results = evaluate_by_segment(
    segment_column="history_group",
    split_name="validation",
    score_column=BASELINE_SCORE_COLUMNS[str(best_validation_baseline["baseline"])],
)

validation_segment_results.sort_values(["k", "segment_value"])

,segment_column,segment_value,k,evaluated_windows,local_evaluated_windows,precision,hit_rate,recall_local,recall_global,ndcg
0,history_group,P0-P50,5,57566,54755,0.332811,0.773443,0.415206,0.307060,0.480196
3,history_group,P50-P90,5,46210,45461,0.387475,0.810582,0.236661,0.175515,0.441376
6,history_group,P90+,5,12133,12024,0.392318,0.807797,0.182382,0.139225,0.434676
1,history_group,P0-P50,10,57566,54755,0.246472,0.844023,0.568276,0.416520,0.512977
4,history_group,P50-P90,10,46210,45461,0.313620,0.885523,0.366989,0.273185,0.429994
7,history_group,P90+,10,12133,12024,0.320399,0.882717,0.287329,0.219322,0.404758
2,history_group,P0-P50,20,57566,54755,0.166443,0.892871,0.719056,0.520608,0.569130
5,history_group,P50-P90,20,46210,45461,0.236696,0.932504,0.532762,0.396779,0.473329
8,history_group,P90+,20,12133,12024,0.248302,0.930190,0.430473,0.329147,0.421612


---

## 7. Registro opcional no MLflow

O registro no MLflow salva uma run por baseline.

Cada run registra todas as metricas para todos os splits e valores de K:

- `train_*_at_k`
- `validation_*_at_k`
- `test_*_at_k`

Para facilitar comparacao visual entre runs, as metricas de validacao tambem sao registradas sem prefixo de split, por exemplo `ndcg_at_10`, `hit_rate_at_10`, `recall_local_at_10`, `recall_global_at_10` e `precision_at_10`.

O dataset nao e enviado inteiro como artefato do MLflow. O diretorio `data/features/temporal_modeling_dataset_v1/` e versionado com DVC, e o MLflow registra os ponteiros necessarios para reproducibilidade:

- `git_commit`
- `dataset_path`
- `dataset_dvc_file`
- `dataset_dvc_md5`
- `dataset_dvc_size`
- `dataset_dvc_nfiles`
- `dataset_dvc_hash`
- `dataset_dvc_out_path`
- `dataset_versioning = dvc`

Essa separacao evita duplicar storage: DVC versiona dados e MLflow versiona experimentos, metricas e artefatos pequenos de resultado.


Para reduzir latencia no tracking remoto, parametros e metricas sao enviados em lote com `mlflow.log_params` e `mlflow.log_metrics`, em vez de varias chamadas individuais por metrica.


In [16]:
def configure_mlflow():
    load_dotenv()
    tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
    assert tracking_uri, "MLFLOW_TRACKING_URI nao configurado."

    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(EXPERIMENT_NAME)

    return tracking_uri


def get_git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=PROJECT_ROOT,
            text=True,
        ).strip()
    except Exception:
        return "unavailable"


def get_dvc_dataset_metadata(dvc_path):
    metadata = {
        "dvc_path": str(dvc_path.relative_to(PROJECT_ROOT)),
        "dvc_md5": "unavailable",
        "dvc_size": "unavailable",
        "dvc_nfiles": "unavailable",
        "dvc_hash": "unavailable",
        "dvc_out_path": "unavailable",
    }

    current_output = False

    for raw_line in dvc_path.read_text().splitlines():
        line = raw_line.strip()

        if line.startswith("- md5:"):
            current_output = True
            metadata["dvc_md5"] = line.split(":", 1)[1].strip()
        elif current_output and line.startswith("size:"):
            metadata["dvc_size"] = line.split(":", 1)[1].strip()
        elif current_output and line.startswith("nfiles:"):
            metadata["dvc_nfiles"] = line.split(":", 1)[1].strip()
        elif current_output and line.startswith("hash:"):
            metadata["dvc_hash"] = line.split(":", 1)[1].strip()
        elif current_output and line.startswith("path:"):
            metadata["dvc_out_path"] = line.split(":", 1)[1].strip()

    return metadata


def build_run_params(baseline_name):
    dvc_metadata = get_dvc_dataset_metadata(TEMPORAL_MODELING_DATASET_DVC_PATH)

    return {
        "notebook": "05-baseline.ipynb",
        "git_commit": get_git_commit(),
        "dataset_path": str(TEMPORAL_MODELING_DATASET_DIR),
        "dataset_dvc_file": dvc_metadata["dvc_path"],
        "dataset_dvc_md5": dvc_metadata["dvc_md5"],
        "dataset_dvc_size": dvc_metadata["dvc_size"],
        "dataset_dvc_nfiles": dvc_metadata["dvc_nfiles"],
        "dataset_dvc_hash": dvc_metadata["dvc_hash"],
        "dataset_dvc_out_path": dvc_metadata["dvc_out_path"],
        "dataset_versioning": "dvc",
        "ranking_unit": "user_window_id",
        "k_values": ",".join(map(str, K_VALUES)),
        "evaluation_splits": ",".join(EVALUATION_SPLITS),
        "primary_metric": PRIMARY_METRIC,
        "primary_k": PRIMARY_K,
        "baseline_name": baseline_name,
    }


def build_run_metrics(run_results):
    metric_names = [
        "ndcg",
        "hit_rate",
        "recall_local",
        "recall_global",
        "precision",
    ]
    metrics = {}

    for row in run_results.itertuples(index=False):
        split_name = str(row.split)
        k = int(row.k)

        metrics[f"{split_name}_evaluated_windows_at_{k}"] = float(row.evaluated_windows)
        metrics[f"{split_name}_local_evaluated_windows_at_{k}"] = float(
            row.local_evaluated_windows
        )

        for metric_name in metric_names:
            metric_value = float(getattr(row, metric_name))
            metrics[f"{split_name}_{metric_name}_at_{k}"] = metric_value

            if split_name == "validation":
                metrics[f"{metric_name}_at_{k}"] = metric_value

    return metrics


def log_baseline_run(baseline_name, results_df):
    run_results = results_df[results_df["baseline"] == baseline_name].copy()

    with mlflow.start_run(run_name=f"temporal_baseline_{baseline_name}"):
        mlflow.log_params(build_run_params(baseline_name))
        mlflow.log_metrics(build_run_metrics(run_results))

        with tempfile.TemporaryDirectory() as tmp_dir:
            results_path = Path(tmp_dir) / "temporal_baseline_results.csv"
            run_results.to_csv(results_path, index=False)
            mlflow.log_artifact(str(results_path), artifact_path="tables")

In [17]:
if RUN_MLFLOW:
    tracking_uri = configure_mlflow()

    for baseline_name in BASELINE_SCORE_COLUMNS:
        log_baseline_run(baseline_name, baseline_results_df)

    print(f"Baselines registrados no MLflow: {tracking_uri}")
else:
    print("Registro no MLflow desativado. Defina RUN_MLFLOW = True para registrar.")

Registro no MLflow desativado. Defina RUN_MLFLOW = True para registrar.


---

## 8. Decisao para a proxima etapa

A escolha do proximo modelo deve continuar usando `ndcg@10` no split `validation`, pois a proxima etapa avalia um ranker treinavel sobre o mesmo conjunto de candidatos.

Para o notebook `06-experiments.ipynb`, a comparacao principal deve priorizar metricas locais de ranking:

1. `ndcg@10`
2. `hit_rate@10`
3. `recall_local@10`
4. `precision@10`

`recall_global@10` fica como metrica de leitura do sistema completo no baseline. Ela nao precisa ser calculada em toda iteracao do MLP, porque adiciona custo e mistura a qualidade do ranker com a cobertura da geracao de candidatos.

O objetivo do MLP sera superar `ordem_gerador_candidatos` em `ndcg@10` sem depender exclusivamente de `candidate_rank` ou `candidate_source`.
